In [0]:
%scala

import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window

// =====================================================
// 1. SPARK SESSION
// =====================================================

val spark = SparkSession.builder()
  .appName("ProjectNexusTesting")
  .getOrCreate()

// =====================================================
// 2. TEST FRAMEWORK
// =====================================================

var passedTests = 0
var failedTests = 0

def runTest(testName: String)(test: => Boolean): Unit = {
  try {
    if (test) {
      println(s"PASS: $testName")
      passedTests += 1
    } else {
      println(s"FAIL: $testName")
      failedTests += 1
    }
  } catch {
    case e: Exception =>
      println(s"ERROR: $testName")
      println(e.getMessage)
      failedTests += 1
  }
}

// =====================================================
// 3. BRONZE LAYER TESTING
// =====================================================

println("\n========== BRONZE LAYER TESTING ==========")

val bronzeTables = Seq(
  "retail.bronze.customers",
  "retail.bronze.products",
  "retail.bronze.orders",
  "retail.bronze.orders_updates",
  "retail.bronze.orders_v2"
)

val metadataColumns = Seq(
  "_ingestion_timestamp",
  "_source_file",
  "_batch_id"
)

bronzeTables.foreach { tableName =>

  runTest(s"Bronze table exists: $tableName") {
    spark.catalog.tableExists(tableName)
  }

  if (spark.catalog.tableExists(tableName)) {

    val df = spark.table(tableName)

    runTest(s"Metadata columns exist: $tableName") {
      metadataColumns.forall(df.columns.contains)
    }

    if (metadataColumns.forall(df.columns.contains)) {

      val nullMetadataCondition = metadataColumns
        .map(c => col(c).isNull)
        .reduce(_ || _)

      runTest(s"No NULL metadata values: $tableName") {
        df.filter(nullMetadataCondition).count() == 0
      }
    }

    runTest(s"Bronze table contains records: $tableName") {
      df.count() > 0
    }

    println(s"$tableName count: ${df.count()}")
  }
}

// =====================================================
// 4. SILVER LAYER TESTING
// =====================================================

println("\n========== SILVER LAYER TESTING ==========")

val customersDF = spark.table("retail.silver.customers")
val productsDF = spark.table("retail.silver.products")
val ordersDF = spark.table("retail.silver.orders")

// CUSTOMER VALIDATIONS

runTest("Customer IDs contain no NULL values") {
  customersDF.filter(col("customer_id").isNull).count() == 0
}

runTest("No duplicate customer IDs") {
  customersDF.groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count() == 0
}

// PRODUCT VALIDATIONS

runTest("Product IDs contain no NULL values") {
  productsDF.filter(col("product_id").isNull).count() == 0
}

runTest("No duplicate product IDs") {
  productsDF.groupBy("product_id")
    .count()
    .filter(col("count") > 1)
    .count() == 0
}

runTest("Product names are populated") {
  productsDF.filter(
    col("product_name").isNull ||
    trim(col("product_name")) === ""
  ).count() == 0
}

runTest("Product categories are populated") {
  productsDF.filter(
    col("category").isNull ||
    trim(col("category")) === ""
  ).count() == 0
}

// Correct column: unit_price

runTest("Product prices are positive and not NULL") {
  productsDF.filter(
    col("unit_price").isNull ||
    col("unit_price") <= 0
  ).count() == 0
}

// ORDER VALIDATIONS

runTest("Order IDs contain no NULL values") {
  ordersDF.filter(col("order_id").isNull).count() == 0
}

runTest("No duplicate order IDs") {
  ordersDF.groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .count() == 0
}

// EMAIL VALIDATION

runTest("Customer emails have valid format") {
  customersDF.filter(
    col("email").isNull ||
    !col("email").rlike(
      "^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$"
    )
  ).count() == 0
}

// FOREIGN KEY VALIDATION

val invalidCustomerOrders = ordersDF.join(
  customersDF.select("customer_id"),
  Seq("customer_id"),
  "left_anti"
)

runTest("No invalid customer references in orders") {
  invalidCustomerOrders.count() == 0
}

val invalidProductOrders = ordersDF.join(
  productsDF.select("product_id"),
  Seq("product_id"),
  "left_anti"
)

runTest("No invalid product references in orders") {
  invalidProductOrders.count() == 0
}

// REJECTION HANDLING

val rejectedProductsDF =
  spark.table("retail.silver.rejected_products")

runTest("Rejected products contain rejection reasons") {
  rejectedProductsDF.filter(
    col("rejection_reason").isNull ||
    trim(col("rejection_reason")) === ""
  ).count() == 0
}

println(s"Rejected product records: ${rejectedProductsDF.count()}")

// =====================================================
// 5. INCREMENTAL LOAD TESTING
// =====================================================

println("\n========== INCREMENTAL LOAD TESTING ==========")

val incomingUpdatesDF =
  spark.table("retail.bronze.orders_updates")

val currentOrdersDF =
  spark.table("retail.silver.orders")

// Check duplicate IDs after incremental load

runTest("No duplicate order IDs after incremental load") {
  currentOrdersDF.groupBy("order_id")
    .count()
    .filter(col("count") > 1)
    .count() == 0
}

// Identify latest incoming version per order

val latestUpdatesWindow = Window
  .partitionBy("order_id")
  .orderBy(col("last_updated").desc)

val latestIncomingUpdates = incomingUpdatesDF
  .withColumn(
    "row_num",
    row_number().over(latestUpdatesWindow)
  )
  .filter(col("row_num") === 1)
  .drop("row_num")

// Identify stale incoming records

val staleUpdates = latestIncomingUpdates.alias("source")
  .join(
    currentOrdersDF.alias("target"),
    col("source.order_id") === col("target.order_id"),
    "inner"
  )
  .filter(
    col("source.last_updated") < col("target.last_updated")
  )

println(
  s"Stale incoming update candidates: ${staleUpdates.count()}"
)

// =====================================================
// 6. GOLD DIMENSION TESTING
// =====================================================

println("\n========== GOLD DIMENSION TESTING ==========")

val dimCustomerDF =
  spark.table("retail.gold.dim_customer")

val dimProductDF =
  spark.table("retail.gold.dim_product")

val dimDateDF =
  spark.table("retail.gold.dim_date")

// CUSTOMER DIMENSION

runTest("Customer dimension has no NULL keys") {
  dimCustomerDF.filter(
    col("customer_key").isNull
  ).count() == 0
}

runTest("Customer dimension has no duplicate keys") {
  dimCustomerDF.groupBy("customer_key")
    .count()
    .filter(col("count") > 1)
    .count() == 0
}

// PRODUCT DIMENSION

runTest("Product dimension has no NULL keys") {
  dimProductDF.filter(
    col("product_key").isNull
  ).count() == 0
}

runTest("Product dimension has no duplicate keys") {
  dimProductDF.groupBy("product_key")
    .count()
    .filter(col("count") > 1)
    .count() == 0
}

// DATE DIMENSION

runTest("Date dimension has no NULL keys") {
  dimDateDF.filter(
    col("date_key").isNull
  ).count() == 0
}

runTest("Date dimension has no duplicate keys") {
  dimDateDF.groupBy("date_key")
    .count()
    .filter(col("count") > 1)
    .count() == 0
}

// =====================================================
// 7. FINAL TEST SUMMARY
// =====================================================

println("\n==========================================")
println("       PROJECT NEXUS TEST SUMMARY")
println("==========================================")

println(s"Passed tests : $passedTests")
println(s"Failed tests : $failedTests")
println(s"Total tests  : ${passedTests + failedTests}")

if (failedTests == 0) {
  println("RESULT: ALL EXECUTED TESTS PASSED")
} else {
  println("RESULT: SOME TESTS FAILED")
}

println("==========================================")


========== BRONZE LAYER TESTING ==========
PASS: Bronze table exists: retail.bronze.customers
PASS: Metadata columns exist: retail.bronze.customers
PASS: No NULL metadata values: retail.bronze.customers
PASS: Bronze table contains records: retail.bronze.customers
retail.bronze.customers count: 42
PASS: Bronze table exists: retail.bronze.products
PASS: Metadata columns exist: retail.bronze.products
PASS: No NULL metadata values: retail.bronze.products
PASS: Bronze table contains records: retail.bronze.products
retail.bronze.products count: 27
PASS: Bronze table exists: retail.bronze.orders
PASS: Metadata columns exist: retail.bronze.orders
PASS: No NULL metadata values: retail.bronze.orders
PASS: Bronze table contains records: retail.bronze.orders
retail.bronze.orders count: 161
PASS: Bronze table exists: retail.bronze.orders_updates
PASS: Metadata columns exist: retail.bronze.orders_updates
PASS: No NULL metadata values: retail.bronze.orders_updates
PASS: Bronze table contains records: retail.bronze.orders_updates
retail.bronze.orders_updates count: 17
PASS: Bronze table exists: retail.bronze.orders_v2
PASS: Metadata columns exist: retail.bronze.orders_v2
PASS: No NULL metadata values: retail.bronze.orders_v2
PASS: Bronze table contains records: retail.bronze.orders_v2
retail.bronze.orders_v2 count: 12

========== SILVER LAYER TESTING ==========
PASS: Customer IDs contain no NULL values
PASS: No duplicate customer IDs
PASS: Product IDs contain no NULL values
PASS: No duplicate product IDs
PASS: Product names are populated
PASS: Product categories are populated
PASS: Product prices are positive and not NULL
PASS: Order IDs contain no NULL values
PASS: No duplicate order IDs
PASS: Customer emails have valid format
PASS: No invalid customer references in orders
PASS: No invalid product references in orders
PASS: Rejected products contain rejection reasons
Rejected product records: 6

========== INCREMENTAL LOAD TESTING ==========
PASS: No duplicate order IDs after incremental load
Stale incoming update candidates: 0

========== GOLD DIMENSION TESTING ==========
PASS: Customer dimension has no NULL keys
PASS: Customer dimension has no duplicate keys
PASS: Product dimension has no NULL keys
PASS: Product dimension has no duplicate keys
PASS: Date dimension has no NULL keys
PASS: Date dimension has no duplicate keys

==========================================
 PROJECT NEXUS TEST SUMMARY
==========================================
Passed tests : 40
Failed tests : 0
Total tests : 40
RESULT: ALL EXECUTED TESTS PASSED
==========================================
import org.apache.spark.sql.{DataFrame, SparkSession}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.expressions.Window
val spark: org.apache.spark.sql.SparkSession = org.apache.spark.sql.classic.SparkSession@338b0e29
var passedTests: Int = 40
var failedTests: Int = 0
def runTest(testName: String)(test: => Boolean): Unit
val bronzeTables: Seq[String] = List(retail.bronze.customers, retail.bronze.products, retail.bronze.orders, retail.bronze.orders_updates, retail.bronze.orders_v2)
val metadataColumns: Seq[String] = List(_ingestion_timestamp, _source_file, _batch_id)
val customersDF: org.apache.spark.sql.DataFrame = [customer_id: string, customer_name: string ... 6 more fields]
val productsDF: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 6 more fields]
val ordersDF: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 10 more fields]
val invalidCustomerOrders: org.apache.spark.sql.DataFrame = [customer_id: string, order_id: string ... 10 more fields]
val invalidProductOrders: org.apache.spark.sql.DataFrame = [product_id: string, order_id: string ... 10 more fields]
val rejectedProductsDF: org.apache.spark.sql.DataFrame = [product_id: string, product_name: string ... 7 more fields]
val incomingUpdatesDF: org.apache.spark.sql.DataFrame = [order_id: string, customer_id: string ... 8 more fields]
val currentOr